In [96]:
!pip install pandas
!pip install matplotlib
!pip install numpy

In [97]:
import pandas as pd
import numpy as np

In [98]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [99]:
aircraft_df = pd.read_csv("raw_data/acftref.csv")
aircraft_df.head()

,CODE,MFR,MODEL,TYPE-ACFT,TYPE-ENG,AC-CAT,BUILD-CERT-IND,NO-ENG,NO-SEATS,AC-WEIGHT,SPEED,TC-DATA-SHEET,TC-DATA-HOLDER,Unnamed: 13
0,0020901,AAR AIRLIFT GROUP INC,UH-60A,6,3,1,0,2,15,CLASS 3,0,,,NaN
1,0030109,EXLINE ACE-C,ACE-C,4,1,1,1,1,1,CLASS 1,82,,,NaN
2,003010D,DELEBAUGH,P,4,1,1,1,1,1,CLASS 1,82,,,NaN
3,003010H,DAL PORTO,BABY ACE D,4,1,1,1,1,1,CLASS 1,82,,,NaN
4,003010P,DUNN,BABY ACE,4,1,1,1,1,1,CLASS 1,82,,,NaN


In [100]:
aircraft_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 93453 entries, 0 to 93452
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   CODE            93453 non-null  str    
 1   MFR             93453 non-null  str    
 2   MODEL           93453 non-null  str    
 3   TYPE-ACFT       93453 non-null  str    
 4   TYPE-ENG        93453 non-null  int64  
 5   AC-CAT          93453 non-null  int64  
 6   BUILD-CERT-IND  93453 non-null  int64  
 7   NO-ENG          93453 non-null  int64  
 8   NO-SEATS        93453 non-null  int64  
 9   AC-WEIGHT       93453 non-null  str    
 10  SPEED           93453 non-null  int64  
 11  TC-DATA-SHEET   93453 non-null  str    
 12  TC-DATA-HOLDER  93453 non-null  str    
 13  Unnamed: 13     0 non-null      float64
dtypes: float64(1), int64(6), str(7)
memory usage: 10.0 MB


In [101]:
# 1. Clean column names first
aircraft_df.columns = aircraft_df.columns.str.strip()

# 2. Identify object columns, then exclude 'CODE'
df_obj_cols = aircraft_df.select_dtypes(['object']).columns
cols_to_strip = df_obj_cols.difference(['CODE'])

# 3. Apply stripping only to the filtered list
aircraft_df[cols_to_strip] = aircraft_df[cols_to_strip].apply(lambda x: x.astype(str).str.strip())

# 4. Drop the ghost column if it exists
if 'Unnamed: 13' in aircraft_df.columns:
    aircraft_df = aircraft_df.drop(columns=['Unnamed: 13'])

aircraft_df.head()

/var/folders/r5/7sdqfmvs4p987n2g0n_nng840000gn/T/ipykernel_46088/1631228276.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df_obj_cols = aircraft_df.select_dtypes(['object']).columns


,CODE,MFR,MODEL,TYPE-ACFT,TYPE-ENG,AC-CAT,BUILD-CERT-IND,NO-ENG,NO-SEATS,AC-WEIGHT,SPEED,TC-DATA-SHEET,TC-DATA-HOLDER
0,0020901,AAR AIRLIFT GROUP INC,UH-60A,6,3,1,0,2,15,CLASS 3,0,,
1,0030109,EXLINE ACE-C,ACE-C,4,1,1,1,1,1,CLASS 1,82,,
2,003010D,DELEBAUGH,P,4,1,1,1,1,1,CLASS 1,82,,
3,003010H,DAL PORTO,BABY ACE D,4,1,1,1,1,1,CLASS 1,82,,
4,003010P,DUNN,BABY ACE,4,1,1,1,1,1,CLASS 1,82,,


In [102]:
cols_to_fix = aircraft_df.columns

# Replace empty/whitespace strings with NaN only in those columns
aircraft_df[cols_to_fix] = aircraft_df[cols_to_fix].replace(r'^\s*$', np.nan, regex=True)

## Split up `CODE`
A code assigned to the aircraft
manufacturer, model and series.

Positions (1-3) - Manufacturer Code

Positions (4-5) - Model Code

Positions (6-7) - Series Code

In [103]:
aircraft_df['MFR_CODE'] = aircraft_df['CODE'].str[0:3]
aircraft_df['MODEL_CODE'] = aircraft_df['CODE'].str[3:5]
aircraft_df['SERIES_CODE'] = aircraft_df['CODE'].str[5:7]

# Displaying the result
print(aircraft_df[['CODE', 'MFR_CODE', 'MODEL_CODE', 'SERIES_CODE']].head())

      CODE MFR_CODE MODEL_CODE SERIES_CODE
0  0020901      002         09          01
1  0030109      003         01          09
2  003010D      003         01          0D
3  003010H      003         01          0H
4  003010P      003         01          0P


In [104]:
aircraft_df

,CODE,MFR,MODEL,TYPE-ACFT,TYPE-ENG,AC-CAT,BUILD-CERT-IND,NO-ENG,NO-SEATS,AC-WEIGHT,SPEED,TC-DATA-SHEET,TC-DATA-HOLDER,MFR_CODE,MODEL_CODE,SERIES_CODE
0,0020901,AAR AIRLIFT GROUP INC,UH-60A,6,3,1,0,2,15,CLASS 3,0,NaN,NaN,002,09,01
1,0030109,EXLINE ACE-C,ACE-C,4,1,1,1,1,1,CLASS 1,82,NaN,NaN,003,01,09
2,003010D,DELEBAUGH,P,4,1,1,1,1,1,CLASS 1,82,NaN,NaN,003,01,0D
3,003010H,DAL PORTO,BABY ACE D,4,1,1,1,1,1,CLASS 1,82,NaN,NaN,003,01,0H
4,003010P,DUNN,BABY ACE,4,1,1,1,1,1,CLASS 1,82,NaN,NaN,003,01,0P
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
93448,9970235,LEO SEKANINA,SAVAGE SHOCK CUB,4,8,1,1,1,2,CLASS 1,0,NaN,NaN,997,02,35
93449,9980000,ZLT ZEPPELIN LUFTSCHIFFTECHNIK,LZ N07-100,3,1,1,0,3,17,CLASS 1,0,NaN,NaN,998,00,00
93450,9980002,ZLT ZEPPELIN LUFTSCHIFFTECHNIK,LZ NO7-101,3,1,1,0,3,15,CLASS 2,0,NaN,NaN,998,00,02
93451,9999999,UNKNOWN,UNKNOWN,1,0,1,1,0,999,CLASS 4,0,NaN,NaN,999,99,99


In [105]:
# Define the mapping based on your requirements
aircraft_type_map = {
    '1': 'Glider',
    '2': 'Balloon',
    '3': 'Blimp/Dirigible',
    '4': 'Fixed wing single engine',
    '5': 'Fixed wing multi engine',
    '6': 'Rotorcraft',
    '7': 'Weight-shift-control',
    '8': 'Powered Parachute',
    '9': 'Gyroplane',
    'H': 'Hybrid Lift',
    'O': 'Other'
}

aircraft_df['TYPE-ACFT'] = aircraft_df['TYPE-ACFT'].astype(str).map(aircraft_type_map)

In [106]:
print(aircraft_df['TYPE-ACFT'].value_counts())

TYPE-ACFT
Fixed wing single engine    71283
Rotorcraft                   9612
Fixed wing multi engine      4742
Glider                       2938
Balloon                      1851
Powered Parachute            1333
Weight-shift-control         1013
Gyroplane                     494
Blimp/Dirigible                94
Hybrid Lift                    81
Other                          12
Name: count, dtype: int64


In [107]:
print(aircraft_df['TYPE-ENG'].value_counts(dropna=False))

TYPE-ENG
1     69984
8      6781
7      5557
0      4005
10     1764
5      1704
2      1375
4      1192
3      1031
11       36
9        17
6         7
Name: count, dtype: int64


In [108]:
engine_type_map = {
    '0': 'None',
    '1': 'Reciprocating',
    '2': 'Turbo-prop',
    '3': 'Turbo-shaft',
    '4': 'Turbo-jet',
    '5': 'Turbo-fan',
    '6': 'Ramjet',
    '7': '2 Cycle',
    '8': '4 Cycle',
    '9': 'Unknown',
    '10': 'Electric',
    '11': 'Rotary'
}

aircraft_df['TYPE-ENG'] = aircraft_df['TYPE-ENG'].astype(str).map(engine_type_map)

In [109]:
print(aircraft_df['TYPE-ENG'].value_counts(dropna=False))

TYPE-ENG
Reciprocating    69984
4 Cycle           6781
2 Cycle           5557
None              4005
Electric          1764
Turbo-fan         1704
Turbo-prop        1375
Turbo-jet         1192
Turbo-shaft       1031
Rotary              36
Unknown             17
Ramjet               7
Name: count, dtype: int64


In [110]:
aircraft_df['AC-CAT'].value_counts(dropna=False)

AC-CAT
1    89674
3     3527
2      252
Name: count, dtype: int64

In [111]:
aircraft_category_map = {
    '1': 'Land',
    '2': 'Sea',
    '3': 'Amphibian',
}

aircraft_df['AC-CAT'] = aircraft_df['AC-CAT'].astype(str).map(aircraft_category_map)

In [112]:
aircraft_df['AC-CAT'].value_counts(dropna=False)

AC-CAT
Land         89674
Amphibian     3527
Sea            252
Name: count, dtype: int64

In [113]:
aircraft_df['BUILD-CERT-IND'].value_counts(dropna=False)

BUILD-CERT-IND
1    78524
0     9301
2     5628
Name: count, dtype: int64

In [114]:
builder_cert_code_map = {
    '0': 'Type Certificated',
    '1': 'Not Type Certificated',
    '2': 'Light Sport'
}

aircraft_df['BUILD-CERT-IND'] = aircraft_df['BUILD-CERT-IND'].astype(str).map(builder_cert_code_map)

In [115]:
aircraft_df['BUILD-CERT-IND'].value_counts(dropna=False)

BUILD-CERT-IND
Not Type Certificated    78524
Type Certificated         9301
Light Sport               5628
Name: count, dtype: int64

In [117]:
aircraft_df['AC-WEIGHT'].value_counts(dropna=False)

AC-WEIGHT
CLASS 1    88284
CLASS 3     2896
CLASS 4     1460
CLASS 2      813
Name: count, dtype: int64

In [119]:
weight_map = {
    'CLASS 1': 'Up to 12,499',
    'CLASS 2': '12,500 - 19,999',
    'CLASS 3': '20,000 and over',
    'CLASS 4': 'UAV up to 55',
}

aircraft_df['AC-WEIGHT'] = aircraft_df['AC-WEIGHT'].astype(str).map(weight_map)

In [123]:
aircraft_df['SPEED'].value_counts(dropna=False)

SPEED
0      79679
85      3421
45      1470
60      1043
112      831
       ...  
382        1
52         1
22         1
24         1
267        1
Name: count, Length: 243, dtype: int64

## Cleanup `SPEED` column
- Replace `0` mph with `NaN`, since 0mph crusing speed is not possible

In [129]:
aircraft_df['SPEED'] = aircraft_df['SPEED'].astype('Int64')
aircraft_df['SPEED'] = aircraft_df['SPEED'].replace(0, np.nan)

In [130]:
print(aircraft_df['SPEED'].value_counts(dropna=False))

SPEED
<NA>    79679
85       3421
45       1470
60       1043
112       831
        ...  
382         1
52          1
22          1
24          1
267         1
Name: count, Length: 243, dtype: Int64


## Column Name Cleaning

In [132]:
aircraft_df.columns = (aircraft_df.columns
                     .str.replace(' ', '_', regex=False)
                     .str.replace('-', '_', regex=False)
                     .str.replace('(', '', regex=False)
                     .str.replace(')', '', regex=False)
                     .str.lower())

In [135]:
# Define the new order
cols = aircraft_df.columns.tolist()

# The columns you want to move
to_move = ['mfr_code', 'model_code', 'series_code']

# Remove them from their current position
for col in to_move:
    cols.remove(col)

# Re-insert them starting at index 1 (right after 'code' at index 0)
for i, col in enumerate(to_move):
    cols.insert(1 + i, col)

# Apply the new order
aircraft_df = aircraft_df[cols]

In [140]:
aircraft_df.to_csv('./clean_data/ACFTREF.csv', index=False, na_rep='NULL')